# 13 GHSL Final Six-City Model and Jaipur Generalization

This Colab notebook continues from Notebooks 11 and 12.

It uses `/content/drive/MyDrive/remote_exec.sh` for every EC2 operation. It:

1. validates the immutable six-city artifacts;
2. retrains the selected Linear Regression pipeline on all 264,600 transitions through 2020;
3. discovers and validates Jaipur without using it for model selection;
4. builds Jaipur longitudinal and transition datasets;
5. evaluates the historical Notebook 12 model on Jaipur 2015→2020;
6. applies the final all-history model to Jaipur's 2020 state as an **experimental 2025 projection**;
7. saves metrics, predictions, hotspot summaries, and manifests.

Run cells sequentially. Every HDFS write refuses to overwrite an existing destination.

In [ ]:
# ============================================================
# CELL 1: DRIVE, PATHS, CONSTANTS
# ============================================================
from google.colab import drive
from pathlib import Path
import base64, json, os, shutil, subprocess, time

drive.mount('/content/drive', force_remount=False)
REMOTE_EXEC=Path('/content/drive/MyDrive/remote_exec.sh')
DRIVE_SSH_KEY=Path('/content/drive/MyDrive/hadoop-hdfs.pem')
COLAB_SSH_KEY=Path('/content/hadoop-hdfs.pem')
INV11=Path('/content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_inventory.json')
MAN12=Path('/content/drive/MyDrive/GHSL_Urban_Growth_Model/model_training_v1/training_manifest.json')
ART=Path('/content/drive/MyDrive/GHSL_Urban_Growth_Model/final_model_and_jaipur_v1')
for p in [REMOTE_EXEC,DRIVE_SSH_KEY,INV11,MAN12]:
    if not p.is_file(): raise FileNotFoundError(str(p))
REMOTE_EXEC.chmod(0o700)
if COLAB_SSH_KEY.exists(): COLAB_SSH_KEY.unlink()
shutil.copy2(DRIVE_SSH_KEY,COLAB_SSH_KEY); COLAB_SSH_KEY.chmod(0o600)
ART.mkdir(parents=True,exist_ok=True)
INV=json.loads(INV11.read_text())
M12=json.loads(MAN12.read_text())
TRANS='/data/ghsl/modeling/six_city_transitions_v1'
EVAL_MODEL='/data/ghsl/modeling/models/six_city_growth_v1/linear_regression'
FINAL_MODEL='/data/ghsl/modeling/models/six_city_growth_final_v1/linear_regression'
JAIPUR_LONG='/data/ghsl/modeling/jaipur_longitudinal_v1'
JAIPUR_TRANS='/data/ghsl/modeling/jaipur_transitions_v1'
JAIPUR_EVAL='/data/ghsl/modeling/jaipur_evaluation_v1'
JAIPUR_PRED='/data/ghsl/modeling/jaipur_predictions_v1'
EXPECTED_EPOCHS=[1975,1980,1985,1990,1995,2000,2005,2010,2015,2020]
print('CELL 1: SUCCESS')

In [ ]:
# ============================================================
# CELL 2: REMOTE EXECUTION HELPER
# ============================================================
def run_remote(command,timeout=300,show_output=True,raise_on_error=True):
    r=subprocess.run([str(REMOTE_EXEC),command],text=True,capture_output=True,timeout=timeout)
    if show_output and r.stdout: print(r.stdout)
    if show_output and r.stderr: print('REMOTE STDERR:'); print(r.stderr)
    if raise_on_error and r.returncode!=0: raise RuntimeError('Remote return code '+str(r.returncode))
    return r
r=run_remote("set -e; echo REMOTE_EXEC_CONNECTION_SUCCESS; whoami; hostname",90)
print('CELL 2: SUCCESS')

In [ ]:
# ============================================================
# CELL 3: HDFS, SPARK, AND PYSPARK ML PREFLIGHT
# ============================================================
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
export SETUPTOOLS_USE_DISTUTILS=local
for d in NameNode DataNode SecondaryNameNode; do jps | awk '{print $2}' | grep -qx "$d"; done
hdfs dfsadmin -safemode get | grep -qi 'Safe mode is OFF'
for p in /data/ghsl/modeling/six_city_transitions_v1 /data/ghsl/modeling/models/six_city_growth_v1/linear_regression; do hdfs dfs -test -e "$p"; done
python3 - <<'PY_REMOTE'
import setuptools,distutils
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer,VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
print('PYSPARK_ML_IMPORTS_PASS')
PY_REMOTE
echo NOTEBOOK_13_PREFLIGHT_SUCCESS
"""
r=run_remote(cmd,300)
if 'NOTEBOOK_13_PREFLIGHT_SUCCESS' not in r.stdout: raise RuntimeError('Preflight failed')
print('CELL 3: SUCCESS')

In [ ]:
# ============================================================
# CELL 4: FREEZE PROVENANCE, FEATURES, AND MODEL SELECTION
# ============================================================
if INV.get('expected_total_rows')!=294000: raise RuntimeError('Notebook 11 inventory mismatch')
if M12.get('jaipur_used_in_training') is not False: raise RuntimeError('Jaipur contamination detected')
R=M12['results']
if R['split_counts']!={'train':205800,'validation':29400,'test':29400}: raise RuntimeError('Notebook 12 split mismatch')
FEATURES=R['features']
if len(FEATURES)!=31: raise RuntimeError('Expected 31 features')
TARGET_SOURCE=M12['target_source_column']
if TARGET_SOURCE!='built_surface_total_m2': raise RuntimeError('Unexpected target source')
feature_contract={'features':FEATURES,'target_source':TARGET_SOURCE,'label':'next_built_surface_total_m2','imputer':'median','assembler':'VectorAssembler'}
(ART/'feature_contract.json').write_text(json.dumps(feature_contract,indent=2,sort_keys=True)+'\n')
print('Selected final model: linear_regression')
print('Features:',len(FEATURES))
print('MODEL_SELECTION_FREEZE_SUCCESS')
print('CELL 4: SUCCESS')

In [ ]:
# ============================================================
# CELL 5: AUDIT ALL 264,600 FINAL-TRAINING EXAMPLES
# ============================================================
payload=base64.b64encode(json.dumps({'input':TRANS,'features':FEATURES}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
P=/tmp/nb13_audit.json; S=/tmp/nb13_audit.py
echo __P__ | base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
import json,sys
p=json.load(open(sys.argv[1])); s=SparkSession.builder.appName('NB13-Final-Training-Audit').getOrCreate(); s.sparkContext.setLogLevel('WARN')
try:
 d=s.read.parquet(p['input']); rows=d.count(); keys=d.select('city_slug','grid_cell_id','from_epoch','to_epoch').distinct().count()
 nulls=d.filter(F.col('label').isNull()|F.isnan('label')).count(); cities=d.select('city_slug').distinct().count()
 epochs={int(r['to_epoch']):r['count'] for r in d.groupBy('to_epoch').count().collect()}
 missing=[x for x in p['features'] if x not in d.columns]
 if rows!=264600 or keys!=264600 or nulls!=0 or cities!=6 or len(epochs)!=9 or missing: raise RuntimeError(repr({'rows':rows,'keys':keys,'nulls':nulls,'cities':cities,'epochs':epochs,'missing':missing}))
 print('FINAL_AUDIT_JSON_BEGIN'); print(json.dumps({'rows':rows,'keys':keys,'cities':cities,'epochs':epochs},sort_keys=True)); print('FINAL_AUDIT_JSON_END')
 print('FINAL_TRAINING_DATA_AUDIT_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 4g --conf spark.ui.enabled=false "$S" "$P"
rm -f "$S" "$P"
""".replace('__P__',payload)
r=run_remote(cmd,3600)
print('CELL 5: SUCCESS')

In [ ]:
# ============================================================
# CELL 6: TRAIN FINAL LINEAR MODEL ON ALL TRANSITIONS THROUGH 2020
# ============================================================
payload=base64.b64encode(json.dumps({'input':TRANS,'output':FINAL_MODEL,'features':FEATURES}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
export SETUPTOOLS_USE_DISTUTILS=local
P=/tmp/nb13_train.json; S=/tmp/nb13_train.py
echo __P__ | base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
from pyspark.ml import Pipeline,PipelineModel
from pyspark.ml.feature import Imputer,VectorAssembler
from pyspark.ml.regression import LinearRegression
import json,sys
p=json.load(open(sys.argv[1])); s=SparkSession.builder.appName('NB13-Final-Linear-Model').getOrCreate(); s.sparkContext.setLogLevel('WARN')
try:
 fs=s._jvm.org.apache.hadoop.fs.FileSystem.get(s._jsc.hadoopConfiguration()); op=s._jvm.org.apache.hadoop.fs.Path(p['output'])
 if fs.exists(op): raise RuntimeError('Refusing overwrite: '+p['output'])
 d=s.read.parquet(p['input']).filter(F.col('label').isNotNull() & (~F.isnan('label')))
 if d.count()!=264600: raise RuntimeError('Training row count mismatch')
 imp=['imp_'+x for x in p['features']]
 pipe=Pipeline(stages=[Imputer(inputCols=p['features'],outputCols=imp,strategy='median'),VectorAssembler(inputCols=imp,outputCol='features',handleInvalid='keep'),LinearRegression(featuresCol='features',labelCol='label',predictionCol='prediction',maxIter=100,regParam=0.01,elasticNetParam=0.0)])
 model=pipe.fit(d); model.write().save(p['output'])
 loaded=PipelineModel.load(p['output']); sample=d.limit(1000); scored=loaded.transform(sample)
 if scored.count()!=1000 or scored.filter(F.col('prediction').isNull()|F.isnan('prediction')).count()!=0: raise RuntimeError('Reload scoring failed')
 lr=model.stages[-1]
 out={'training_rows':264600,'model_path':p['output'],'intercept':lr.intercept,'coefficients':list(lr.coefficients),'features':p['features']}
 print('FINAL_MODEL_JSON_BEGIN'); print(json.dumps(out,sort_keys=True)); print('FINAL_MODEL_JSON_END'); print('FINAL_SIX_CITY_LINEAR_MODEL_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=24 "$S" "$P"
rm -f "$S" "$P"
""".replace('__P__',payload)
r=run_remote(cmd,7200,show_output=False)
if 'FINAL_SIX_CITY_LINEAR_MODEL_SUCCESS' not in r.stdout:
 print(r.stdout[-12000:]); print(r.stderr[-8000:]); raise RuntimeError('Final training failed')
FINAL_MODEL_INFO=json.loads(r.stdout.split('FINAL_MODEL_JSON_BEGIN',1)[1].split('FINAL_MODEL_JSON_END',1)[0].strip())
(ART/'final_model_manifest.json').write_text(json.dumps(FINAL_MODEL_INFO,indent=2,sort_keys=True)+'\n')
print(json.dumps(FINAL_MODEL_INFO,indent=2)[:5000]); print('CELL 6: SUCCESS')

In [ ]:
# ============================================================
# CELL 7: READ-ONLY JAIPUR HDFS DISCOVERY
# ============================================================
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
hdfs dfs -find /data/ghsl -iname '*jaipur*' 2>/dev/null | head -1000
echo JAIPUR_HDFS_DISCOVERY_SUCCESS
"""
r=run_remote(cmd,900)
paths=[x.strip() for x in r.stdout.splitlines() if x.strip().endswith('/enriched_grid_year')]
paths=sorted(set(paths))
print('Candidate enriched roots:',paths)
if not paths: raise RuntimeError('No Jaipur enriched_grid_year root found. Process Jaipur before continuing.')
JAIPUR_CANDIDATES=paths
print('CELL 7: SUCCESS')

In [ ]:
# ============================================================
# CELL 8: RESOLVE ONE COMPLETE JAIPUR ROOT
# ============================================================
checks=[]
for root in JAIPUR_CANDIDATES:
    c=run_remote("source /home/ubuntu/ghsl-sedona-env.sh; export HADOOP_HOME=/usr/local/hadoop; export PATH=$HADOOP_HOME/bin:$PATH; hdfs dfs -ls '"+root+"' 2>/dev/null | sed -n 's#.*epoch=##p'",300,show_output=False,raise_on_error=False)
    epochs=sorted({int(x.strip().split('/')[0]) for x in c.stdout.splitlines() if x.strip().split('/')[0].isdigit()})
    checks.append({'root':root,'epochs':epochs})
complete=[x for x in checks if x['epochs']==EXPECTED_EPOCHS]
standard=[x for x in complete if '/curated/cities/city=jaipur/' in x['root']]
selected=standard or complete
print(json.dumps(checks,indent=2))
if len(selected)!=1: raise RuntimeError('Could not uniquely resolve one complete Jaipur root: '+repr(selected))
JAIPUR_ROOT=selected[0]['root']
print('JAIPUR_ENRICHED_ROOT =',JAIPUR_ROOT)
print('JAIPUR_INPUT_RESOLUTION_SUCCESS')
print('CELL 8: SUCCESS')

In [ ]:
# ============================================================
# CELL 9: AUDIT JAIPUR AND BUILD 49,000-ROW LONGITUDINAL DATASET
# ============================================================
payload=base64.b64encode(json.dumps({'root':JAIPUR_ROOT,'out':JAIPUR_LONG,'epochs':EXPECTED_EPOCHS,'features':FEATURES}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
P=/tmp/nb13_jlong.json; S=/tmp/nb13_jlong.py
echo __P__ | base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
from functools import reduce
import json,sys
p=json.load(open(sys.argv[1])); s=SparkSession.builder.appName('NB13-Jaipur-Longitudinal').getOrCreate(); s.sparkContext.setLogLevel('WARN')
try:
 fs=s._jvm.org.apache.hadoop.fs.FileSystem.get(s._jsc.hadoopConfiguration()); op=s._jvm.org.apache.hadoop.fs.Path(p['out'])
 if fs.exists(op): raise RuntimeError('Refusing overwrite: '+p['out'])
 frames=[]
 for e in p['epochs']:
  d=s.read.parquet(p['root']+'/epoch='+str(e)); missing=[x for x in p['features']+["built_surface_total_m2","grid_cell_id"] if x not in d.columns]
  if missing: raise RuntimeError('Missing Jaipur fields '+repr(missing))
  if d.count()!=4900 or d.select('grid_cell_id').distinct().count()!=4900: raise RuntimeError('Jaipur epoch validation failed '+str(e))
  for c in ['built_observed_grid_area_m2','built_coverage_fraction']:
   d=d.withColumn(c,F.col(c).cast('double'))
  for c in ['epoch','city_name','city_slug']:
   if c in d.columns:d=d.drop(c)
  frames.append(d.withColumn('city_name',F.lit('Jaipur')).withColumn('city_slug',F.lit('jaipur')).withColumn('epoch',F.lit(e).cast('int')))
 all=reduce(lambda a,b:a.unionByName(b),frames)
 if all.count()!=49000 or all.select('epoch','grid_cell_id').distinct().count()!=49000: raise RuntimeError('Jaipur longitudinal validation failed')
 all.repartition('epoch').write.mode('errorifexists').partitionBy('epoch').parquet(p['out'])
 if s.read.parquet(p['out']).count()!=49000: raise RuntimeError('Jaipur reread failed')
 print('JAIPUR_LONGITUDINAL_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false "$S" "$P"
rm -f "$S" "$P"
""".replace('__P__',payload)
r=run_remote(cmd,7200)
print('CELL 9: SUCCESS')

In [ ]:
# ============================================================
# CELL 10: BUILD 44,100 JAIPUR TRANSITIONS
# ============================================================
payload=base64.b64encode(json.dumps({'input':JAIPUR_LONG,'out':JAIPUR_TRANS}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
P=/tmp/nb13_jtrans.json; S=/tmp/nb13_jtrans.py
echo __P__ | base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F,Window
import json,sys
p=json.load(open(sys.argv[1])); s=SparkSession.builder.appName('NB13-Jaipur-Transitions').getOrCreate(); s.sparkContext.setLogLevel('WARN')
try:
 fs=s._jvm.org.apache.hadoop.fs.FileSystem.get(s._jsc.hadoopConfiguration()); op=s._jvm.org.apache.hadoop.fs.Path(p['out'])
 if fs.exists(op): raise RuntimeError('Refusing overwrite: '+p['out'])
 d=s.read.parquet(p['input']); w=Window.partitionBy('city_slug','grid_cell_id').orderBy('epoch')
 t=(d.withColumn('to_epoch',F.lead('epoch').over(w)).withColumn('label',F.lead(F.col('built_surface_total_m2').cast('double')).over(w)).withColumn('current_built_surface_m2',F.col('built_surface_total_m2').cast('double')).withColumn('built_surface_growth_m2',F.col('label')-F.col('current_built_surface_m2')).withColumnRenamed('epoch','from_epoch').filter((F.col('to_epoch').isNotNull())&((F.col('to_epoch')-F.col('from_epoch'))==5)))
 if t.count()!=44100 or t.select('grid_cell_id','from_epoch','to_epoch').distinct().count()!=44100: raise RuntimeError('Jaipur transition validation failed')
 t.repartition('to_epoch').write.mode('errorifexists').partitionBy('to_epoch').parquet(p['out'])
 print('JAIPUR_TRANSITIONS_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false "$S" "$P"
rm -f "$S" "$P"
""".replace('__P__',payload)
r=run_remote(cmd,7200)
print('CELL 10: SUCCESS')

In [ ]:
# ============================================================
# CELL 11: HONEST JAIPUR 2015->2020 EVALUATION
# Uses Notebook 12 evaluation model, not the all-history model.
# ============================================================
payload=base64.b64encode(json.dumps({'input':JAIPUR_TRANS,'model':EVAL_MODEL,'out':JAIPUR_EVAL}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"; export SETUPTOOLS_USE_DISTUTILS=local
P=/tmp/nb13_jeval.json; S=/tmp/nb13_jeval.py
echo __P__ | base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
import json,sys
p=json.load(open(sys.argv[1])); s=SparkSession.builder.appName('NB13-Jaipur-Heldout-Evaluation').getOrCreate(); s.sparkContext.setLogLevel('WARN')
try:
 fs=s._jvm.org.apache.hadoop.fs.FileSystem.get(s._jsc.hadoopConfiguration()); op=s._jvm.org.apache.hadoop.fs.Path(p['out'])
 if fs.exists(op): raise RuntimeError('Refusing overwrite: '+p['out'])
 d=s.read.parquet(p['input']).filter(F.col('to_epoch')==2020)
 if d.count()!=4900: raise RuntimeError('Expected 4900 Jaipur test rows')
 m=PipelineModel.load(p['model']); pred=m.transform(d).withColumn('error',F.col('prediction')-F.col('label')).withColumn('abs_error',F.abs('error')).withColumn('persistence_prediction',F.col('current_built_surface_m2'))
 def met(df,col):
  x=df.withColumn('prediction_eval',F.col(col)); return {k:RegressionEvaluator(labelCol='label',predictionCol='prediction_eval',metricName=k).evaluate(x) for k in ['mae','rmse','r2']}
 result={'model':met(pred,'prediction'),'persistence':met(pred,'persistence_prediction'),'rows':pred.count()}
 pred.select('grid_cell_id','from_epoch','to_epoch','label','current_built_surface_m2','prediction','persistence_prediction','error','abs_error','grid_1km_row','grid_1km_col','grid_min_x','grid_max_x','grid_min_y','grid_max_y').write.mode('errorifexists').parquet(p['out'])
 print('JAIPUR_METRICS_JSON_BEGIN'); print(json.dumps(result,sort_keys=True)); print('JAIPUR_METRICS_JSON_END'); print('JAIPUR_HISTORICAL_GENERALIZATION_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false "$S" "$P"
rm -f "$S" "$P"
""".replace('__P__',payload)
r=run_remote(cmd,7200,show_output=False)
if 'JAIPUR_HISTORICAL_GENERALIZATION_SUCCESS' not in r.stdout:
 print(r.stdout[-12000:]);print(r.stderr[-8000:]);raise RuntimeError('Jaipur evaluation failed')
JAIPUR_METRICS=json.loads(r.stdout.split('JAIPUR_METRICS_JSON_BEGIN',1)[1].split('JAIPUR_METRICS_JSON_END',1)[0].strip())
(ART/'jaipur_historical_metrics.json').write_text(json.dumps(JAIPUR_METRICS,indent=2,sort_keys=True)+'\n')
print(json.dumps(JAIPUR_METRICS,indent=2));print('CELL 11: SUCCESS')

In [ ]:
# ============================================================
# CELL 12: EXPERIMENTAL 2025 JAIPUR PROJECTION + HOTSPOTS
# ============================================================
payload=base64.b64encode(json.dumps({'input':JAIPUR_LONG,'model':FINAL_MODEL,'out':JAIPUR_PRED,'features':FEATURES}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"; export SETUPTOOLS_USE_DISTUTILS=local
P=/tmp/nb13_jpred.json; S=/tmp/nb13_jpred.py
echo __P__ | base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel
import json,sys
p=json.load(open(sys.argv[1])); s=SparkSession.builder.appName('NB13-Jaipur-Experimental-2025').getOrCreate(); s.sparkContext.setLogLevel('WARN')
try:
 fs=s._jvm.org.apache.hadoop.fs.FileSystem.get(s._jsc.hadoopConfiguration()); op=s._jvm.org.apache.hadoop.fs.Path(p['out'])
 if fs.exists(op): raise RuntimeError('Refusing overwrite: '+p['out'])
 d=s.read.parquet(p['input']).filter(F.col('epoch')==2020).withColumnRenamed('epoch','from_epoch')
 if d.count()!=4900: raise RuntimeError('Expected 4900 Jaipur 2020 rows')
 m=PipelineModel.load(p['model']); q=m.transform(d)
 q=(q.withColumn('raw_prediction',F.col('prediction')).withColumn('constrained_prediction',F.greatest(F.lit(0.0),F.least(F.col('prediction'),F.col('built_observed_grid_area_m2')))).withColumn('predicted_growth_m2_raw',F.col('raw_prediction')-F.col('built_surface_total_m2')).withColumn('predicted_growth_m2_constrained',F.col('constrained_prediction')-F.col('built_surface_total_m2')).withColumn('constraint_applied',F.abs(F.col('raw_prediction')-F.col('constrained_prediction'))>1e-9))
 q=q.withColumn('growth_percentile',F.percent_rank().over(Window.orderBy('predicted_growth_m2_constrained'))).withColumn('hotspot_top_10pct',F.col('growth_percentile')>=0.90).withColumn('hotspot_top_5pct',F.col('growth_percentile')>=0.95).withColumn('projection_status',F.lit('experimental_2025_projection')).withColumn('to_epoch',F.lit(2025))
 q.select('grid_cell_id','from_epoch','to_epoch','built_surface_total_m2','raw_prediction','constrained_prediction','predicted_growth_m2_raw','predicted_growth_m2_constrained','constraint_applied','growth_percentile','hotspot_top_10pct','hotspot_top_5pct','grid_1km_row','grid_1km_col','grid_min_x','grid_max_x','grid_min_y','grid_max_y','projection_status').write.mode('errorifexists').parquet(p['out'])
 summary={'rows':q.count(),'top10':q.filter('hotspot_top_10pct').count(),'top5':q.filter('hotspot_top_5pct').count(),'constrained':q.filter('constraint_applied').count()}
 print('JAIPUR_PROJECTION_JSON_BEGIN');print(json.dumps(summary,sort_keys=True));print('JAIPUR_PROJECTION_JSON_END');print('JAIPUR_FINAL_MODEL_SCORING_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false "$S" "$P"
rm -f "$S" "$P"
""".replace('__P__',payload)
r=run_remote(cmd,7200,show_output=False)
if 'JAIPUR_FINAL_MODEL_SCORING_SUCCESS' not in r.stdout:
 print(r.stdout[-12000:]);print(r.stderr[-8000:]);raise RuntimeError('Projection failed')
JAIPUR_PROJECTION=json.loads(r.stdout.split('JAIPUR_PROJECTION_JSON_BEGIN',1)[1].split('JAIPUR_PROJECTION_JSON_END',1)[0].strip())
print(JAIPUR_PROJECTION);print('CELL 12: SUCCESS')

In [ ]:
# ============================================================
# CELL 13: EXPORT COMPACT CSV RESULTS TO GOOGLE DRIVE
# ============================================================
REMOTE_EXPORT='/tmp/nb13_exports'; run_remote("rm -rf "+REMOTE_EXPORT+"; mkdir -p "+REMOTE_EXPORT,120,show_output=False)
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
S=/tmp/nb13_export.py
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession
s=SparkSession.builder.appName('NB13-Export').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 s.read.parquet('/data/ghsl/modeling/jaipur_evaluation_v1').coalesce(1).write.mode('overwrite').option('header',True).csv('/tmp/nb13_exports/eval')
 s.read.parquet('/data/ghsl/modeling/jaipur_predictions_v1').coalesce(1).write.mode('overwrite').option('header',True).csv('/tmp/nb13_exports/projection')
finally:s.stop()
PY_REMOTE
spark-submit --master local[1] --driver-memory 3g --conf spark.ui.enabled=false "$S"
rm -f "$S"
"""
run_remote(cmd,3600,show_output=False)
# Copy via remote_exec streaming to Colab, then Drive.
for remote_dir,name in [('eval','jaipur_2015_2020_predictions.csv'),('projection','jaipur_experimental_2025_predictions.csv')]:
    r=run_remote("cat $(find /tmp/nb13_exports/"+remote_dir+" -name 'part-*.csv' | head -1)",600,show_output=False)
    (ART/name).write_text(r.stdout,encoding='utf-8')
print('Exports:',sorted(p.name for p in ART.glob('*.csv')))
print('CELL 13: SUCCESS')

In [ ]:
# ============================================================
# CELL 14: FINAL PROJECT MANIFEST AND COMPLETION GATE
# ============================================================
manifest={
 'created_at_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
 'training_cities':INV['training_city_slugs'],
 'jaipur_used_in_model_selection':False,
 'jaipur_used_in_six_city_training':False,
 'evaluation_model':EVAL_MODEL,
 'final_model':FINAL_MODEL,
 'final_training_rows':264600,
 'target_source':TARGET_SOURCE,
 'feature_count':len(FEATURES),
 'jaipur_source_root':JAIPUR_ROOT,
 'jaipur_longitudinal':JAIPUR_LONG,
 'jaipur_transitions':JAIPUR_TRANS,
 'jaipur_historical_evaluation':JAIPUR_EVAL,
 'jaipur_experimental_projection':JAIPUR_PRED,
 'jaipur_metrics':JAIPUR_METRICS,
 'projection_summary':JAIPUR_PROJECTION,
 'limitations':[
  'The final model was trained on six-city transitions through 2020.',
  'Jaipur 2015-2020 historical evaluation uses the earlier Notebook 12 evaluation model.',
  'The 2025 Jaipur output is temporal extrapolation and has no observed 2025 validation label.',
  'High R2 for next built surface partly reflects strong temporal persistence.'
 ]
}
FINAL_MANIFEST=ART/'final_project_manifest.json'
FINAL_MANIFEST.write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
checks=run_remote("source /home/ubuntu/ghsl-sedona-env.sh; export HADOOP_HOME=/usr/local/hadoop; export PATH=$HADOOP_HOME/bin:$PATH; for p in '"+EVAL_MODEL+"' '"+FINAL_MODEL+"' '"+JAIPUR_LONG+"' '"+JAIPUR_TRANS+"' '"+JAIPUR_EVAL+"' '"+JAIPUR_PRED+"'; do hdfs dfs -test -e \"$p\" || exit 1; done; echo FINAL_HDFS_CHECKS_PASS",300)
if not FINAL_MANIFEST.is_file(): raise RuntimeError('Final manifest missing')
print('Six-city evaluation model preserved: YES')
print('Final six-city model trained through 2020: YES')
print('Jaipur used during model selection: NO')
print('Jaipur used in six-city training: NO')
print('Jaipur historical generalization evaluated: YES')
print('Experimental Jaipur projection generated: YES')
print('Evaluation and deployment claims separated: YES')
print('HDFS source datasets modified: NO')
print('NOTEBOOK_13_FINAL_MODEL_AND_JAIPUR_SUCCESS')
print('CELL 14: SUCCESS')

## Interpretation guardrails

- Report Notebook 12 metrics as the six-city historical backtest.
- Report Cell 11 as the honest unseen-city Jaipur 2015→2020 evaluation.
- Treat Cell 12 only as an experimental 2025 projection.
- Never describe the 2025 projection as observed, verified, or causal.